# K-Nearest Neighbors (KNN) Algorithm
 
This notebook demonstrates the K-Nearest Neighbors (KNN) algorithm for classification. We'll use synthetic data to illustrate both a naive and a vectorized implementation, and compare their performance.

- $\mathbf{X} \in \mathbb{R}^{N \times D}$: Training data (N samples, D features)
- $\mathbf{y} \in \mathbb{R}^{N}$: Training labels
- $\mathbf{Z} \in \mathbb{R}^{M \times D}$: Test data (M samples, D features)

In [1]:
import time
from collections import Counter
import numpy as np

In [2]:
# Number of training samples, features, and test samples
num_train_samples = 500
num_features = 20
num_test_samples = 4
k_neighbors = 5

In [3]:
# Generate random training and test data
X_train = np.random.random((num_train_samples, num_features))
Z_test = np.random.random((num_test_samples, num_features))
y_train = np.random.randint(3, size=num_train_samples)

## KNN Classification: Naive Approach
For each test sample $z_j$, compute the Euclidean distance to every training sample $x_i$ and select the $k$ nearest neighbors.

$$ \underset{i}{\operatorname{argmin}}\; \|x_i - z_j\|_2 $$

In [4]:
def knn_naive(X_train, y_train, Z_test, k):
    """
    Naive KNN implementation using explicit loops.
    Args:
        X_train: Training data, shape (N, D)
        y_train: Training labels, shape (N,)
        Z_test: Test data, shape (M, D)
        k: Number of neighbors
    Returns:
        List of predicted labels for test data.
    """
    num_train = X_train.shape[0]
    num_test = Z_test.shape[0]
    predictions = []
    for j in range(num_test):
        distances = np.zeros(num_train)
        for i in range(num_train):
            distances[i] = np.linalg.norm(X_train[i, :] - Z_test[j, :], 2)
        nearest_indices = np.argsort(distances)[:k]
        most_common = Counter(y_train[nearest_indices]).most_common(1)[0][0]
        predictions.append(most_common)
    return predictions

## KNN Classification: Vectorized Approach
To speed up distance computation, we can use the following identity:
$$\|x_i - z_j\|_2 = \sqrt{(x_i - z_j)^T (x_i-z_j)} = \sqrt{x_i^T x_i -2 x_i^T z_j + z_j^T z_j} $$

- $\operatorname{diag}(X X^T)$ gives $x_i^T x_i$ for all $i$
- $X Z^T$ gives $x_i^T z_j$ for all $i, j$
- $\operatorname{diag}(Z Z^T)$ gives $z_j^T z_j$ for all $j$

In [13]:
def knn_vectorized(X_train, y_train, Z_test, k):
    """
    Vectorized KNN implementation using matrix operations.
    Args:
        X_train: Training data, shape (N, D)
        y_train: Training labels, shape (N,)
        Z_test: Test data, shape (M, D)
        k: Number of neighbors
    Returns:
        List of predicted labels for test data.
    """
    num_train = X_train.shape[0]
    num_test = Z_test.shape[0]
    # Compute squared norms
    X_norms = np.diag(np.matmul(X_train, X_train.T))
    Z_norms = np.diag(np.matmul(Z_test, Z_test.T))
    
    # X_norms = np.tile(X_norms, (num_test, 1)).T 
    # Z_norms = np.tile(Z_norms, (num_train, 1))
    XZ_norms = np.matmul(X_train, Z_test.T)
    # Compute distance matrix (N, M)
    distances = np.sqrt(
      X_norms[:,None]  - 2 * XZ_norms  + Z_norms [None,:]
    )
    predictions = []
    for j in range(num_test):
        nearest_indices = np.argsort(distances[:, j])[:k]
        most_common = Counter(y_train[nearest_indices]).most_common(1)[0][0]
        predictions.append(most_common)
    return predictions

In [14]:
# Timing the naive KNN implementation
start_time = time.time()
predictions_naive = knn_naive(X_train, y_train, Z_test, k_neighbors)
print(f"Naive KNN time: {time.time() - start_time:.4f} seconds")

Naive KNN time: 0.0336 seconds


In [15]:
# Timing the vectorized KNN implementation
start_time = time.time()
predictions_vectorized = knn_vectorized(X_train, y_train, Z_test, k_neighbors)
print(f"Vectorized KNN time: {time.time() - start_time:.4f} seconds")

Vectorized KNN time: 147.0716 seconds


In [8]:
# Check that both implementations give the same result
print("Predictions match:", predictions_naive == predictions_vectorized)

Predictions match: True
